# LACCI complete experiments notebook

This notebook runs the full paper experiment suite with the patched codebase:

- main recommendation-target runs under **max aggregation**
- `tfidf_meta`, `minilm_meta`, and `meta_only`
- robustness runs under **mean aggregation**
- task-group CV only
- SHAP artifacts saved with explicit `row_ids`

Outputs are written to separate result directories and include CSV summaries, SHAP artifacts, and metadata files.


In [ ]:
from pathlib import Path
import json
import importlib
import shutil
import pandas as pd
import joblib

import open_flow_minimal_CC18_shap_metrics_v4 as wf

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## Configuration

In [ ]:
# Core paths
CACHE_DIR = "minimal_cache_cc18_v4"
RESULTS_DIR_MAX = "minimal_results_cc18_lacci_max_v4"
RESULTS_DIR_MEAN = "minimal_results_cc18_lacci_mean_v4"

# OpenML / experiment settings
FUNCTION = "predictive_accuracy"
CV_FOLDS = 5
RANDOM_STATE = 42

# SHAP settings
COMPUTE_SHAP = True
SHAP_BACKGROUND_SIZE = 200
SHAP_EVAL_SIZE = 100
COMPUTE_SHAP_INTERACTIONS = False
SHAP_INTERACTION_EVAL_SIZE = 50

# Cache resets
CLEAR_RESULTS = False
CLEAR_CACHE = False

# Paper model restriction
SELECTED_MODELS = {"hist_gbrt", "random_forest"}

# Paper experiments
MAX_CONFIGS = [
    {"name": "tfidf_meta", "text_mode": "tfidf", "use_task_metafeatures": True},
    {"name": "minilm_meta", "text_mode": "minilm", "use_task_metafeatures": True},
    {"name": "meta_only", "text_mode": "none", "use_task_metafeatures": True},
]

MEAN_CONFIGS = [
    {"name": "tfidf_meta", "text_mode": "tfidf", "use_task_metafeatures": True},
    {"name": "minilm_meta", "text_mode": "minilm", "use_task_metafeatures": True},
]


In [ ]:
for p in [RESULTS_DIR_MAX, RESULTS_DIR_MEAN]:
    if CLEAR_RESULTS and Path(p).exists():
        shutil.rmtree(p)

if CLEAR_CACHE and Path(CACHE_DIR).exists():
    shutil.rmtree(CACHE_DIR)

cache = wf.DiskCache(CACHE_DIR)
print("Cache dir:", cache.cache_dir.resolve())
print("Results max:", Path(RESULTS_DIR_MAX).resolve())
print("Results mean:", Path(RESULTS_DIR_MEAN).resolve())


## Restrict models to the paper

In [ ]:
_original_get_models = wf.get_models

def get_models_paper_only(random_state=42):
    models = _original_get_models(random_state=random_state)
    return {k: v for k, v in models.items() if k in SELECTED_MODELS}

wf.get_models = get_models_paper_only
print("Using models:", list(wf.get_models(random_state=RANDOM_STATE).keys()))


## Load CC18 tasks and evaluations

In [ ]:
bundle = wf.load_cc18_from_openml(
    function=FUNCTION,
    suite_name="OpenML-CC18",
    cache=cache,
)

tasks_df = bundle["tasks_df"]
evals_df = bundle["evaluations_df"]

print("tasks_df shape:", tasks_df.shape)
print("evals_df shape:", evals_df.shape)
display(tasks_df.head())
display(evals_df.head())


## Fetch / cache only the flow records actually used

In [ ]:
used_flow_ids = sorted(set(pd.to_numeric(evals_df["flow_id"], errors="coerce").dropna().astype(int).tolist()))
print("Unique flow ids in evaluations:", len(used_flow_ids))

flows_cache_file = Path(CACHE_DIR) / "flows_used_from_evals_v4.joblib"

if flows_cache_file.exists():
    flows = joblib.load(flows_cache_file)
else:
    import openml
    flows = {}
    for i, fid in enumerate(used_flow_ids, start=1):
        try:
            flow = openml.flows.get_flow(int(fid))
            flows[int(fid)] = {
                "id": int(fid),
                "name": getattr(flow, "name", ""),
                "full_name": getattr(flow, "full_name", ""),
                "class_name": getattr(flow, "class_name", ""),
                "external_version": getattr(flow, "external_version", ""),
                "version": getattr(flow, "version", ""),
                "parameters": getattr(flow, "parameters", None),
                "components": getattr(flow, "components", None),
                "uploader": getattr(flow, "uploader", None),
            }
        except Exception as e:
            print(f"[{i}/{len(used_flow_ids)}] failed flow {fid}: {e}")
        if i % 100 == 0 or i == len(used_flow_ids):
            print(f"Fetched {i}/{len(used_flow_ids)} flows...")
    joblib.dump(flows, flows_cache_file)

print("Cached flows:", len(flows))
list(flows.items())[:1]


## Run helper

In [ ]:
def run_configs(configs, agg_mode, results_dir):
    outputs = {}
    results_dir = Path(results_dir)
    results_dir.mkdir(parents=True, exist_ok=True)

    for cfg in configs:
        run_dir = results_dir / cfg["name"]
        run_dir.mkdir(parents=True, exist_ok=True)

        print(f"Running {agg_mode} :: {cfg['name']} ...")
        out = wf.run_cc18_pipeline(
            flows=flows,
            tasks_df=tasks_df,
            evals_df=evals_df,
            agg_mode=agg_mode,
            text_mode=cfg["text_mode"],
            use_task_metafeatures=cfg["use_task_metafeatures"],
            run_row_kfold=False,
            run_task_group_kfold=True,
            cv_folds=CV_FOLDS,
            random_state=RANDOM_STATE,
            compute_shap=COMPUTE_SHAP,
            shap_background_size=SHAP_BACKGROUND_SIZE,
            shap_eval_size=SHAP_EVAL_SIZE,
            compute_shap_interactions=COMPUTE_SHAP_INTERACTIONS,
            shap_interaction_eval_size=SHAP_INTERACTION_EVAL_SIZE,
            cache_dir=CACHE_DIR,
            results_dir=run_dir,
        )

        meta = {
            "name": cfg["name"],
            "agg_mode": agg_mode,
            "text_mode": cfg["text_mode"],
            "use_task_metafeatures": cfg["use_task_metafeatures"],
            "cv_folds": CV_FOLDS,
            "random_state": RANDOM_STATE,
            "models": list(wf.get_models(random_state=RANDOM_STATE).keys()),
            "results_dir": str(run_dir),
        }
        (run_dir / "run_metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
        outputs[cfg["name"]] = out
        print("Done:", cfg["name"])
    return outputs


## Run max-aggregation paper experiments

In [ ]:
outputs_max = run_configs(MAX_CONFIGS, agg_mode="max", results_dir=RESULTS_DIR_MAX)
list(outputs_max.keys())


## Run mean-aggregation robustness experiments

In [ ]:
outputs_mean = run_configs(MEAN_CONFIGS, agg_mode="mean", results_dir=RESULTS_DIR_MEAN)
list(outputs_mean.keys())


## Quick inspection of max summaries

In [ ]:
for name, out in outputs_max.items():
    print("=" * 80)
    print(name)
    display(out["summary_df"])


## Quick inspection of mean summaries

In [ ]:
for name, out in outputs_mean.items():
    print("=" * 80)
    print(name)
    display(out["summary_df"])


## Build consolidated CSVs for paper notebooks

In [ ]:
def consolidate_anchor(results_root: Path, out_csv: Path):
    frames = []
    for meta_file in results_root.rglob("run_metadata.json"):
        meta = json.loads(meta_file.read_text(encoding="utf-8"))
        run_dir = meta_file.parent
        anchor_file = run_dir / "predictive_anchor.csv"
        if anchor_file.exists():
            df = pd.read_csv(anchor_file)
            df["experiment"] = meta["name"]
            frames.append(df)
    if frames:
        out = pd.concat(frames, ignore_index=True)
        out.to_csv(out_csv, index=False)
        return out
    return pd.DataFrame()

def consolidate_local(results_root: Path, out_csv: Path):
    frames = []
    for meta_file in results_root.rglob("run_metadata.json"):
        meta = json.loads(meta_file.read_text(encoding="utf-8"))
        run_dir = meta_file.parent
        local_file = run_dir / "local_shap_metrics_summary.csv"
        if local_file.exists():
            df = pd.read_csv(local_file)
            df["experiment"] = meta["name"]
            frames.append(df)
    if frames:
        out = pd.concat(frames, ignore_index=True)
        out.to_csv(out_csv, index=False)
        return out
    return pd.DataFrame()

max_root = Path(RESULTS_DIR_MAX)
mean_root = Path(RESULTS_DIR_MEAN)

anchor_max = consolidate_anchor(max_root, max_root / "lacci_predictive_anchor.csv")
local_max = consolidate_local(max_root, max_root / "lacci_local_shap_metrics_summary.csv")

anchor_mean = consolidate_anchor(mean_root, mean_root / "lacci_predictive_anchor.csv")
local_mean = consolidate_local(mean_root, mean_root / "lacci_local_shap_metrics_summary.csv")

print("Saved consolidated files:")
print(max_root / "lacci_predictive_anchor.csv")
print(max_root / "lacci_local_shap_metrics_summary.csv")
print(mean_root / "lacci_predictive_anchor.csv")
print(mean_root / "lacci_local_shap_metrics_summary.csv")


## Sanity check SHAP row identifiers

In [ ]:
for run_name in ["tfidf_meta", "minilm_meta"]:
    pkl = Path(RESULTS_DIR_MAX) / run_name / "shap_artifacts.pkl"
    if not pkl.exists():
        continue
    artifacts = joblib.load(pkl)
    fold0 = artifacts["task_group_kfold"]["hist_gbrt"]["shap_artifacts"][0]
    print(run_name, "row_ids sample:", fold0.row_ids[:5])


## Final notes

This notebook produces separate directories for max and mean runs and writes metadata files for each experiment. The figures notebook can read these consolidated CSVs and SHAP artifacts directly.